<a href="https://colab.research.google.com/github/Atovenish/THO.github.io/blob/main/HandsOn-2-2-LikelihoodFitting.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Only if you are using colab, you need to mount your drive and go to the directory where the necessary files are located : StatsDataAna

In [31]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [61]:
!find /content/drive/ -name "StatsDataAna" -type d
# Thường đường dẫn đúng sẽ là: /content/drive/MyDrive/StatsDataAna (không có dấu cách ở MyDrive)
cd /content/drive/MyDrive/StatsDataAna

[Errno 2] No such file or directory: 'content/drive/mydrive/StatsDataAna'
/content


Additionally, you need to install ROOT software in Colab

In [36]:
!pip install root

Binned maximum likelihood fitting is much faster than unbinned maximum likelihood fitting.

We will use histograms as PDF yielding n = (n1,n2,n3,...,nN) in N bins.

In [37]:
#we will rely on ROOT data analysis framework for fitting
import ROOT

In [38]:
from ROOT import TChain, TFile, TH1F, TCanvas
from math import *

names = ["ttbar","wjets","dy","data"]

for name in names:
    # open the file
    chain = TChain("events");
    chain.Add('files/'+name+'.root')

    f = TFile('hist_'+name+'.root', 'RECREATE')

    h_njets = TH1F('h_njets','number of jets',10,0.0,10.0)


    entries = chain.GetEntries()
    for i in range(entries):
        chain.GetEntry(i)
        #met = math.sqrt( chain.MET_px**2 + chain.MET_py**2 ) # we can use missing energy to remove Z boson events
        if chain.NMuon > 1:
            h_njets.Fill(chain.NJet, chain.EventWeight)

    ntotal = h_njets.Integral()
    print ("total number of events from ", name , "= ", "%.1f" % ntotal, "(raw events = ", entries , ")")

    f.Write()
    f.Close()

Error in <TFile::TFile>: file /content/files/ttbar.root does not exist
Error in <TFile::TFile>: file /content/files/wjets.root does not exist
Error in <TFile::TFile>: file /content/files/dy.root does not exist
Error in <TFile::TFile>: file /content/files/data.root does not exist


total number of events from  ttbar =  0.0 (raw events =  0 )
total number of events from  wjets =  0.0 (raw events =  0 )
total number of events from  dy =  0.0 (raw events =  0 )
total number of events from  data =  0.0 (raw events =  0 )


In [ ]:
ls *.root

##### answer :

```
hist_data.root  hist_dy.root  hist_ttbar.root  hist_wjets.root
```

Now we will read the histograms.

In [ ]:
from ROOT import TChain, TFile, TH1F, TCanvas

f_data = TFile("hist_data.root")
f_dy = TFile("hist_dy.root")
f_wjets = TFile("hist_wjets.root")
f_ttbar = TFile("hist_ttbar.root")

h_data = f_data.Get("h_njets")
h_dy = f_dy.Get("h_njets")
h_wjets = f_wjets.Get("h_njets")
h_ttbar = f_ttbar.Get("h_njets")

And check how many bins we have in the histogram.

In [ ]:
nbins = h_data.GetXaxis().GetNbins()
print (nbins)

##### answer:

```
10
```

We will use RooStats for the binned likelihood fitting. We don't need to worry about this part.
We will just import necessary library and set the variable and histograms in this RooStats format.  

In [ ]:
from ROOT import RooRealVar, RooDataHist, RooArgList
x = RooRealVar("x","x", 0, 10)
data = RooDataHist("data","data", ROOT.RooArgList(x), h_data)
ttbar = RooDataHist("ttbar","ttbar", ROOT.RooArgList(x), h_ttbar)
dy = RooDataHist("zjets","zjets", ROOT.RooArgList(x), h_dy)
wjets = RooDataHist("wjets","wjets", ROOT.RooArgList(x), h_wjets)

We will need to make templates for each process.

In [ ]:
from ROOT import RooHistPdf
pdf_ttbar = RooHistPdf("pdf_ttbar","pdf_ttbar", ROOT.RooArgSet(x), ttbar)
pdf_dy = RooHistPdf("pdf_dy","pdf_dy", ROOT.RooArgSet(x), dy)
pdf_wjets = RooHistPdf("pdf_wjets","pdf_wjets", ROOT.RooArgSet(x), wjets)

create parameter and initialize it

In [ ]:
nttbar = RooRealVar("nttbar","nttbar",200, 0, 10000)
ndy = RooRealVar("ndy","ndy", 20000 , 0, 40000)
nwjets = RooRealVar("nwjets","nwjets", 300 , 0, 10000)

create a model

extended likelihood model below

$$ M(x) = N_{ttbar} \cdot S_{ttbar} (x) + N_{dy} \cdot B_{dy} (x) + N_{wjets} \cdot B_{wjets} (x)  $$

In [ ]:
from ROOT import RooAddPdf
model = RooAddPdf("model","model",RooArgList(pdf_ttbar, pdf_dy, pdf_wjets), RooArgList(nttbar, ndy, nwjets))

perform the fit

In [ ]:
fitResult = model.fitTo( data );

We will visualize the fit result. Each histogram will have the post fit number for normalization.

In [ ]:
c = TCanvas("c","c",1)
xframe = x.frame()
data.plotOn(xframe)
model.paramOn(xframe, ROOT.RooFit.Layout(0.65,0.9,0.9) )
model.plotOn(xframe,ROOT.RooFit.Components("pdf_wjets,pdf_dy,pdf_ttbar"),ROOT.RooFit.LineColor(2),ROOT.RooFit.FillColor(2),ROOT.RooFit.DrawOption("F"))
model.plotOn(xframe,ROOT.RooFit.Components("pdf_wjets,pdf_dy"),ROOT.RooFit.LineColor(5),ROOT.RooFit.FillColor(5),ROOT.RooFit.DrawOption("F"))
model.plotOn(xframe,ROOT.RooFit.Components("pdf_wjets"),ROOT.RooFit.LineColor(3),ROOT.RooFit.FillColor(3),ROOT.RooFit.DrawOption("F"))
model.plotOn(xframe)
data.plotOn(xframe)
xframe.Draw()
c.SetLogy()
c.Draw()

##### task

Can you wriate a macro to fit again including single_top process this time?

### Thực hiện Fit bao gồm Single Top

Chúng ta sẽ nạp file `hist_single_top.root`, tạo PDF cho nó và cập nhật mô hình `RooAddPdf`.

In [ ]:
from ROOT import RooHistPdf, RooArgList, RooRealVar, RooAddPdf, TFile

# 1. Đọc file single_top
f_single_top = TFile("hist_single_top.root")
h_single_top = f_single_top.Get("h_njets")

# 2. Tạo RooDataHist và RooHistPdf cho single_top
single_top_datahist = RooDataHist("single_top_dh", "single_top_dh", RooArgList(x), h_single_top)
pdf_single_top = RooHistPdf("pdf_single_top", "pdf_single_top", ROOT.RooArgSet(x), single_top_datahist)

# 3. Tạo biến số lượng sự kiện cho single_top
nsingletop = RooRealVar("nsingletop", "nsingletop", 100, 0, 5000)

# 4. Cập nhật Model bao gồm cả 4 thành phần
model_extended = RooAddPdf("model_extended", "model with single top",
                           RooArgList(pdf_ttbar, pdf_dy, pdf_wjets, pdf_single_top),
                           RooArgList(nttbar, ndy, nwjets, nsingletop))

# 5. Tiến hành fit
fitResult_ext = model_extended.fitTo(data)


In [ ]:
# 6. Vẽ kết quả
c2 = TCanvas("c2", "Fit with Single Top", 800, 600)
xframe_ext = x.frame()
data.plotOn(xframe_ext)
model_extended.plotOn(xframe_ext, ROOT.RooFit.Components("pdf_ttbar,pdf_dy,pdf_wjets,pdf_single_top"), ROOT.RooFit.FillColor(ROOT.kRed), ROOT.RooFit.DrawOption("F"))
model_extended.plotOn(xframe_ext, ROOT.RooFit.Components("pdf_dy,pdf_wjets,pdf_single_top"), ROOT.RooFit.FillColor(ROOT.kYellow), ROOT.RooFit.DrawOption("F"))
model_extended.plotOn(xframe_ext, ROOT.RooFit.Components("pdf_wjets,pdf_single_top"), ROOT.RooFit.FillColor(ROOT.kGreen), ROOT.RooFit.DrawOption("F"))
model_extended.plotOn(xframe_ext, ROOT.RooFit.Components("pdf_single_top"), ROOT.RooFit.FillColor(ROOT.kBlue), ROOT.RooFit.DrawOption("F"))
model_extended.plotOn(xframe_ext)
data.plotOn(xframe_ext)

xframe_ext.Draw()
c2.SetLogy()
c2.Draw()

### Giải quyết lỗi lệnh `cd`
Lỗi `No such file or directory` thường do đường dẫn không chính xác. Bạn hãy thử chạy lệnh sau để tìm chính xác thư mục `StatsDataAna` đang nằm ở đâu:

In [60]:
!find /content/drive/ -name "StatsDataAna" -type d
# Thường đường dẫn đúng sẽ là: /content/drive/MyDrive/StatsDataAna (không có dấu cách ở MyDrive)